# Install the required libraries

In [ ]:
!pip install -q -U transformers accelerate
!pip install -q peft trl bitsandbytes datasets scikit-learn

#Import the Python libraries

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
from tqdm.notebook import tqdm

# Load Qwen3.5-2B-Base with 4-bit quantization

In [ ]:
model_id = "Qwen/Qwen3.5-0.8B"

bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
  model_id,
  quantization_config=bnb_config,
  device_map="auto",
  dtype=torch.bfloat16
)

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

# Load the tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.eos_token_id

# Load the wolof dataset

In [ ]:
dataset = load_dataset("csv", data_files="/content/wolof_centralized_corpus.csv")
dataset = dataset["train"]

In [ ]:
dataset[0]

{'text': 'bataaxal bii jëwriñu lu ajju ci mbiru bitim réew bu ekuwatër moo ko wara wóoral, te dafa wara mengoo ak yenni càkuteef yi.'}

# Split train and test data

In [ ]:
train_dataset = dataset.select(range(1000))
test_dataset = dataset.select(range(1000, 1200))

In [ ]:
train_dataset[0]

{'text': 'bataaxal bii jëwriñu lu ajju ci mbiru bitim réew bu ekuwatër moo ko wara wóoral, te dafa wara mengoo ak yenni càkuteef yi.'}

# Configuring LoRA Adapters

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        'q_proj','k_proj','v_proj','o_proj',
        'gate_proj','up_proj','down_proj'
    ]
)

# Define the training configuration

In [ ]:
training_args = SFTConfig(
    output_dir="./qwen35-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    weight_decay=0.01,
    optim="adamw_torch",
    fp16=True,
    eval_strategy="epoch",
    num_train_epochs=2,
    dataset_text_field="text",
)

# Train the model

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_config,
    args=training_args,
    processing_class=tokenizer
)
trainer.train()

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


Step,Training Loss
10,5.726101
20,5.263938
30,4.968589
40,4.821759
50,4.856244
60,4.694866
70,4.714730
80,4.659988
90,4.657224
100,4.581992


TrainOutput(global_step=250, training_loss=4.461882995605468, metrics={'train_runtime': 1477.5531, 'train_samples_per_second': 1.354, 'train_steps_per_second': 0.169, 'total_flos': 346744983379200.0, 'train_loss': 4.461882995605468})

# Save the trained adapter

In [ ]:
trainer.model.save_pretrained("qwen35-small-news-class")
tokenizer.save_pretrained("qwen35-small-news-class")

('qwen35-small-news-class/tokenizer_config.json',
 'qwen35-small-news-class/chat_template.jinja',
 'qwen35-small-news-class/tokenizer.json')

# Inference Model

In [ ]:

trainer.model.eval()

prompt = "Jox ñatti xeeti feem ngir am wérgi-yaram."

messages = [{"role": "user", "content": prompt}]

input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

with torch.inference_mode():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
)
print(response)

Daxaay, ci wàllu wérgi-yaram, joxe ñetti xeeti feem ngir am wérgi-yaram.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
save_path = "/content/drive/MyDrive/qwen35-wolof_Next_token prediction"

trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Modèle sauvegardé dans : {save_path}")

Modèle sauvegardé dans : /content/drive/MyDrive/qwen35-wolof_Next_token prediction


# Evaluate the fine-tuned model

In [ ]:
from transformers import AutoModelForCausalLM
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.bfloat16
)


from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    "/kaggle/working/qwen35-small-news-class"
)

model.eval()
model.print_trainable_parameters()